# RetainIQ — Phase 9.1: Knowledge Source Inventory & Corpus Construction

## Objective

I assemble the RAG knowledge corpus from the reporting views created in Phases 6–8 and a curated project-methodology reference. I preserve source, topic, and record-level metadata so retrieved evidence remains traceable.

### Notebook presentation rule

For every relevant analytical code cell, I place a **Result & conclusion** markdown cell immediately after it. The analytical code also prints a concise `Result:` line based on the executed data. Setup-only cells are not padded with artificial conclusions.

In [1]:
from pathlib import Path
from getpass import getpass
import os
import pandas as pd
import numpy as np
import mysql.connector
from mysql.connector import Error
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

ROOT = Path('..')
OUTPUT_DIR = ROOT / 'outputs'
KNOWLEDGE_DIR = ROOT / 'knowledge'
CONFIG_DIR = ROOT / 'config'
OUTPUT_DIR.mkdir(exist_ok=True)

MYSQL_CONFIG = {
    'host': 'localhost',
    'port': 3306,
    'user': 'retainiq_user',
    'password': getpass('Enter MySQL password for retainiq_user: '),
    'database': 'retainiq',
}

def print_result(message):
    print(f'Result: {message}')

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)

def run_query(query):
    connection = get_connection()
    cursor = connection.cursor(dictionary=True)
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        return pd.DataFrame(rows)
    finally:
        cursor.close()
        connection.close()

In [2]:
methodology_path = KNOWLEDGE_DIR / 'project_methodology.md'
methodology_text = methodology_path.read_text(encoding='utf-8')

print_result(f'Loaded the project methodology reference with {len(methodology_text):,} characters.')
print(methodology_text[:1_200])

Result: Loaded the project methodology reference with 2,744 characters.
# RetainIQ — Project Knowledge Reference

## Project purpose
RetainIQ is a telecom customer-retention analytics project. I move from customer-level data quality and warehouse modeling through churn analysis, segmentation, geographic analysis, retention strategy, and benchmark context. This reference is intended to give the GenAI/RAG layer stable definitions and interpretation rules.

## Phase map
1. Business Understanding & Data Quality Audit
2. Data Cleaning & Transformation
3. MySQL Data Modeling & Analytical Warehouse Design
4. EDA & Statistical Modeling
5. Customer Segmentation & Retention Profiling
6. Geospatial & Market-Level Retention Analysis
7. Retention Strategy & Business Intelligence Layer
8. Benchmarking & Performance Context
9. GenAI / RAG Knowledge Layer

## Core definitions

### Churn rate
`Churn Rate = Churned Customers / Total Customers × 100`

### Revenue at risk
Revenue at risk is a **historical

### Result & conclusion

I now have a stable project-level reference containing the definitions, assumptions, and interpretation rules that the RAG layer should preserve.

In [3]:
view_manifest = pd.DataFrame([
    {'source_name': 'vw_retention_strategy_priority', 'source_type': 'phase_07_strategy', 'topic': 'retention_strategy'},
    {'source_name': 'vw_retention_impact_scenarios', 'source_type': 'phase_07_strategy', 'topic': 'retention_strategy'},
    {'source_name': 'vw_retention_action_framework', 'source_type': 'phase_07_strategy', 'topic': 'retention_strategy'},
    {'source_name': 'vw_market_retention', 'source_type': 'phase_06_geography', 'topic': 'geography'},
    {'source_name': 'vw_market_segment_risk', 'source_type': 'phase_06_geography', 'topic': 'geography'},
    {'source_name': 'vw_geographic_risk_grid', 'source_type': 'phase_06_geography', 'topic': 'geography'},
    {'source_name': 'vw_benchmark_reference', 'source_type': 'phase_08_benchmarking', 'topic': 'benchmarking'},
    {'source_name': 'vw_segment_benchmark_gap', 'source_type': 'phase_08_benchmarking', 'topic': 'benchmarking'},
    {'source_name': 'vw_state_benchmark_gap', 'source_type': 'phase_08_benchmarking', 'topic': 'benchmarking'},
    {'source_name': 'vw_city_benchmark_gap', 'source_type': 'phase_08_benchmarking', 'topic': 'benchmarking'},
    {'source_name': 'vw_benchmark_gap_summary', 'source_type': 'phase_08_benchmarking', 'topic': 'benchmarking'},
    {'source_name': 'vw_city_benchmark_management', 'source_type': 'phase_08_benchmarking', 'topic': 'benchmarking'},
    {'source_name': 'vw_segment_strategy_benchmark', 'source_type': 'phase_08_benchmarking', 'topic': 'benchmarking'},
])

def try_load_view(view_name):
    try:
        df = run_query(f'SELECT * FROM {view_name}')
        return df, None
    except Exception as exc:
        return pd.DataFrame(), str(exc)

inventory_rows=[]
loaded_frames={}
for item in view_manifest.to_dict('records'):
    df, error = try_load_view(item['source_name'])
    loaded_frames[item['source_name']] = df
    inventory_rows.append({
        **item,
        'available': error is None,
        'rows': len(df),
        'columns': len(df.columns),
        'error': error or ''
    })

source_inventory = pd.DataFrame(inventory_rows)
display(source_inventory)

,source_name,source_type,topic,available,rows,columns,error
0,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,True,1110,16,
1,vw_retention_impact_scenarios,phase_07_strategy,retention_strategy,True,4436,6,
2,vw_retention_action_framework,phase_07_strategy,retention_strategy,True,3,11,
3,vw_market_retention,phase_06_geography,geography,True,1106,10,
4,vw_market_segment_risk,phase_06_geography,geography,True,61,9,
5,vw_geographic_risk_grid,phase_06_geography,geography,True,1624,8,
6,vw_benchmark_reference,phase_08_benchmarking,benchmarking,True,6,9,
7,vw_segment_benchmark_gap,phase_08_benchmarking,benchmarking,True,3,13,
8,vw_state_benchmark_gap,phase_08_benchmarking,benchmarking,True,1,11,
9,vw_city_benchmark_gap,phase_08_benchmarking,benchmarking,True,27,13,


### Result & conclusion

The source inventory now shows exactly which Phase 6–8 reporting views are available. Missing views are recorded rather than silently replaced with invented data.

In [4]:
def clean_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value

def row_to_text(source_name, topic, row):
    parts = [f'Source: {source_name}', f'Topic: {topic}']
    for key, value in row.items():
        value = clean_value(value)
        if value is not None and str(value).strip() != '':
            parts.append(f'{key}: {value}')
    return ' | '.join(parts)

records=[]
for item in source_inventory.to_dict('records'):
    source_name = item['source_name']
    if not item['available']:
        continue
    df = loaded_frames[source_name]
    for row_number, row in df.iterrows():
        text = row_to_text(source_name, item['topic'], row)
        records.append({
            'document_id': f"{source_name}:{row_number}",
            'source_name': source_name,
            'source_type': item['source_type'],
            'topic': item['topic'],
            'record_number': int(row_number),
            'text': text,
        })

records.append({
    'document_id': 'project_methodology:0',
    'source_name': 'project_methodology',
    'source_type': 'curated_project_reference',
    'topic': 'definitions',
    'record_number': 0,
    'text': methodology_text,
})

knowledge_documents = pd.DataFrame(records)
knowledge_documents.to_csv(OUTPUT_DIR / 'knowledge_documents.csv', index=False)
source_inventory.to_csv(OUTPUT_DIR / 'knowledge_source_inventory.csv', index=False)

print_result(f'Created {len(knowledge_documents):,} knowledge records from {knowledge_documents.source_name.nunique():,} sources.')
display(knowledge_documents.head(8))

Result: Created 8,411 knowledge records from 14 sources.


,document_id,source_name,source_type,topic,record_number,text
0,vw_retention_strategy_priority:0,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,0,Source: vw_retention_strategy_priority | Topic...
1,vw_retention_strategy_priority:1,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,1,Source: vw_retention_strategy_priority | Topic...
2,vw_retention_strategy_priority:2,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,2,Source: vw_retention_strategy_priority | Topic...
3,vw_retention_strategy_priority:3,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,3,Source: vw_retention_strategy_priority | Topic...
4,vw_retention_strategy_priority:4,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,4,Source: vw_retention_strategy_priority | Topic...
5,vw_retention_strategy_priority:5,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,5,Source: vw_retention_strategy_priority | Topic...
6,vw_retention_strategy_priority:6,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,6,Source: vw_retention_strategy_priority | Topic...
7,vw_retention_strategy_priority:7,vw_retention_strategy_priority,phase_07_strategy,retention_strategy,7,Source: vw_retention_strategy_priority | Topic...


### Result & conclusion

I now have a portable knowledge corpus combining structured Phase 6–8 evidence with the project's curated definitions and interpretation rules. Each record retains its source and topic so later retrieval can be audited.

In [5]:
source_summary = (
    knowledge_documents.groupby(['source_type', 'topic'], as_index=False)
    .agg(records=('document_id', 'count'))
    .sort_values('records', ascending=False)
)
display(source_summary)
print_result(f'The corpus spans {knowledge_documents.topic.nunique()} topic areas: {", ".join(sorted(knowledge_documents.topic.unique()))}.')

,source_type,topic,records
2,phase_07_strategy,retention_strategy,5549
1,phase_06_geography,geography,2791
3,phase_08_benchmarking,benchmarking,70
0,curated_project_reference,definitions,1


Result: The corpus spans 4 topic areas: benchmarking, definitions, geography, retention_strategy.


### Result & conclusion

The corpus is distributed across strategy, geography, benchmarking, and definitions. This separation will help retrieval stay grounded in the relevant project layer rather than relying on a single undifferentiated document.